In [1]:
from mpramnist.Sahu2022 import SahuDataset
from mpramnist.Sahu2022 import LitModel_Sahu
from mpramnist.Sahu2022 import LitModel_Sahu_binary_legnet

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

from mpramnist import transforms as t

import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L
from torch.nn import functional as F

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
)
import matplotlib.pyplot as plt

def plot_pr_auc(model, trainer, loader, figsize=(4, 3)):
    """
    Plot Precision-Recall curve and compute PR-AUC metrics.

    Args:
        model: Trained PyTorch model
        loader: DataLoader with test/validation data
        threshold: Decision threshold for precision/recall calculation
        figsize: Size of the output figure
    """
    sigmoid = nn.Sigmoid()

    # Get predictions
    predictions = trainer.predict(model, dataloaders=loader)

    # Get logits
    logits = torch.cat([pred["predicted"] for pred in predictions]).cpu()
    targets = torch.cat([pred["target"] for pred in predictions]).cpu()

    # Turn to probabilities
    y_scores = sigmoid(logits).numpy()  # probabilities [0, 1]
    y_true = targets.numpy()

    # Diagnostic info
    print(f"Class distribution: {np.unique(y_true, return_counts=True)}")
    print(f"Score range after sigmoid: [{y_scores.min():.3f}, {y_scores.max():.3f}]")

    # Calculate metrics
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    avg_precision = average_precision_score(y_true, y_scores)

    # Plot with more detail
    plt.figure(figsize=figsize)
    plt.plot(recall, precision, label=f"PR Curve (AP={avg_precision:.3f})", linewidth=2)

    # Add baseline (random classifier)
    positive_ratio = y_true.mean()
    plt.axhline(
        y=positive_ratio,
        color="r",
        linestyle="--",
        label=f"Random (AP={positive_ratio:.2f})",
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

# Tasks

In [ ]:
tasks = [
    "RandomEnhancer",  # 0
    "GenomicPromoter",  # 1
    "CapturePromoter",  # 2
    "GenomicEnhancer",  # 3
    "AtacSeq",  # 4
    "Binary",  # 5
]

# Random Enhancer

In [ ]:
BATCH_SIZE = 1024
NUM_WORKERS = 103
train_transform = t.Compose([t.Seq2Tensor()])
val_test_transform = t.Compose([t.Seq2Tensor()])
task = "RandomEnhancer"

train_dataset = SahuDataset(task=task, split="train", transform=train_transform, root="../data")
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
in_channels = 4

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[32, 64, 128, 128, 256, 512, 256],
    pool_sizes=[1, 2, 1, 2, 1, 2, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Sahu(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=2,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    enable_model_summary=False,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, trainer, test_loader)

# Genomic promoter

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 103

train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.Seq2Tensor()])

task = "genomicpromoter"
train_dataset = SahuDataset(task=task, split="train", transform=train_transform, root="../data/")
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data/")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data/")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
in_channels = 4

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[32, 64, 128, 128, 256, 512, 256],
    pool_sizes=[1, 2, 1, 2, 1, 2, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Sahu(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=10,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, test_loader)

# Promoter capture

In [20]:
BATCH_SIZE = 512
NUM_WORKERS = 103
train_transform = t.Compose([t.Seq2Tensor()])
val_test_transform = t.Compose([t.Seq2Tensor()])

In [ ]:
task = "CapturePromoter"

train_dataset = SahuDataset(task=task, split="train", transform=train_transform, root="../data/")
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data/")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data/")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
in_channels = 4

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[32, 64, 128, 128, 256, 512, 256],
    pool_sizes=[1, 2, 1, 2, 1, 2, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Sahu(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=10,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False,
)

# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, test_loader)

# Genomic enhancer

In [ ]:
BATCH_SIZE = 1024
NUM_WORKERS = 103
train_transform = t.Compose([t.Seq2Tensor()])
val_test_transform = t.Compose([t.Seq2Tensor()])

task = "GenomicEnhancer"

train_dataset = SahuDataset(task=task, split="train", transform=train_transform, root="../data/")
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data/")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data/")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
in_channels = 4

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[32, 64, 128, 128, 256, 512, 256],
    pool_sizes=[1, 2, 1, 2, 1, 2, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Sahu(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=10,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False,
)

# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, test_loader)

# ATACseq

In [ ]:
BATCH_SIZE = 2048
NUM_WORKERS = 103
train_transform = t.Compose([t.Seq2Tensor()])
val_test_transform = t.Compose([t.Seq2Tensor()])

task = "atacseq"

train_dataset = SahuDataset(task=task, split="train", transform=train_transform, root="../data/")
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data/")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data/")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
in_channels = 4

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[32, 64, 128, 128, 256, 512, 256],
    pool_sizes=[1, 2, 1, 2, 1, 2, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Sahu(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=1,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False,
)

# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, test_loader)

# Binary task

In [2]:
class HumanLegNetBinary(HumanLegNet):
    def __init__(
        self,
    ):
        super().__init__(
            in_ch=4,
            output_dim=1,
            stem_ch=64,
            stem_ks=11,
            ef_ks=9,
            ef_block_sizes=[80, 96, 112, 128],
            pool_sizes=[2, 2, 2, 2],
            resize_factor=4,
            activation=nn.SiLU,
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.main(x)
        x = self.mapper(x)
        x = F.adaptive_avg_pool1d(x, 1)
        x = x.squeeze(-1)  # without head
        # x = self.head(x)
        # x = x.squeeze(-1)
        # head will be used in trainer because paired task has paired sequences
        return x

In [3]:
BATCH_SIZE = 1024
NUM_WORKERS = 103

train_transform = t.Compose([t.Seq2Tensor(),])
val_test_transform = t.Compose([t.Seq2Tensor()])

binary_train = [None, "promoter_from_input", "enhancer_permutated", "enhancer_from_input"]
task = "binary"

train_dataset = SahuDataset(task=task,binary_class=None,split="train",transform=train_transform,root="../data/",)
val_dataset = SahuDataset(task=task, split="val", transform=val_test_transform, root="../data/")
test_dataset = SahuDataset(task=task, split="test", transform=val_test_transform, root="../data/")

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_dataset), len(val_dataset), len(test_dataset))

using train
using val
using test
2991302 748828 1252044


In [ ]:
model = HumanLegNetBinary()
model.apply(initialize_weights)

seq_model = LitModel_Sahu_binary_legnet(model=model,loss=torch.nn.BCEWithLogitsLoss(),weight_decay=1e-1,lr=1e-3,print_each=1,)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False,
)

# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

trainer.test(seq_model, dataloaders=test_loader)
plot_pr_auc(seq_model, test_loader)